In [ ]:
import os
from langchain_groq import ChatGroq
from tavily import TavilyClient
from dotenv import load_dotenv
from pprint import pprint
from rich import print
load_dotenv()

In [ ]:
model="llama-3.3-70b-versatile"

In [ ]:
llm = ChatGroq(model=model)

In [ ]:
llm.invoke("What is the weather in Saint Louis today?")

In [ ]:
tavily =  TavilyClient()

In [ ]:
query="Who will be Mayor of BMC?"

In [ ]:
response = tavily.search(query=query)

In [ ]:
print(response['results'][0]['content'])

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages

In [ ]:
class State(TypedDict):

    messages: Annotated[list, add_messages]

In [ ]:
def TaskPilot(state:State):
    return {'messages':[llm.invoke(state['messages'])]}

In [ ]:
@tool
def search_web_tool(query:str):
    "Use this tool to search the internext for real-time information"
    tavily_search = TavilyClient()
    response = tavily_search.search(query=query)

    return [
        {'title':r.get('title','No Title available'),
         'content': r.get('content', 'No Content available'),
         'url': r.get('url','No url available')}
         for r in response.get('results',[])
    ]

    

In [ ]:
response = search_web_tool.invoke("Who is Mayor of Gadhinglaj?")

In [ ]:
tools = [search_web_tool]
llm_with_tool = llm.bind_tools(tools)

memory = MemorySaver()

In [ ]:
# Node defincation
def tool_calling_llm(state:State):
    return {'messages':[llm_with_tool.invoke(state['messages'])]}

In [ ]:
# Graph
builder = StateGraph(State)

# Add Nodes
builder.add_node('tool_calling_llm', tool_calling_llm)
builder.add_node('tools',ToolNode(tools))

# Add Edges
builder.add_edge(START,'tool_calling_llm')
builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition
)
builder.add_edge('tools','tool_calling_llm')
graph = builder.compile(checkpointer=memory)
graph

In [ ]:
config = {"configurable":{"thread_id":"user"}}

In [ ]:
response = graph.invoke({"messages":"What is the weather tomorrow in Saint Louis?"}, config=config)
print(response["messages"][-1].content)